In [ ]:
from __future__ import annotations

import json
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms


# ============================================================
# CONFIGURATION
# ============================================================


DATASET_ROOT = Path("data")
MODEL_PATH = Path("models/cifar10_alexnet10.pth")

OUTPUT_DIRECTORY = Path("models/quantized")
OUTPUT_MODEL_PATH = OUTPUT_DIRECTORY / "cifar_alexnet_int8.npz"
OUTPUT_CONFIG_PATH = OUTPUT_DIRECTORY / "quantization_config.json"

BATCH_SIZE = 128
CALIBRATION_BATCHES = 40

INPUT_SCALE = 1.0 / 127.0

CLASS_NAMES = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
]


# ============================================================
# ALEXNET MODEL
# ============================================================


class CifarAlexNet(nn.Module):
    def __init__(self, number_of_classes: int = 10) -> None:
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 8, 5, stride=1, padding=2),
            nn.ReLU(inplace=False),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(8, 16, 3, stride=1, padding=1),
            nn.ReLU(inplace=False),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(16, 32, 3, stride=1, padding=1),
            nn.ReLU(inplace=False),

            nn.Conv2d(32, 32, 3, stride=1, padding=1),
            nn.ReLU(inplace=False),

            nn.Conv2d(32, 16, 3, stride=1, padding=1),
            nn.ReLU(inplace=False),

            nn.MaxPool2d(2, 2),
        )

        self.classifier = nn.Sequential(
            nn.Linear(256, 64),
            nn.ReLU(inplace=False),

            nn.Linear(64, 32),
            nn.ReLU(inplace=False),

            nn.Linear(32, number_of_classes),
        )

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        features = self.features(inputs)
        flattened = torch.flatten(features, start_dim=1)

        return self.classifier(flattened)


# ============================================================
# CALIBRATION
# ============================================================

def calculate_symmetric_scale(minimum: float, maximum: float) -> float:
    maximum_absolute = max(abs(minimum), abs(maximum))

    if maximum_absolute == 0.0:
        return 1.0

    return maximum_absolute / 127.0


def run_calibration(
    model: nn.Module,
    device: torch.device,
) -> tuple[dict[str, float], dict[str, float]]:
    transform = transforms.Compose(
        [
            transforms.ToTensor(),
            transforms.Normalize(
                mean=(0.5, 0.5, 0.5),
                std=(0.5, 0.5, 0.5),
            ),
        ]
    )

    test_dataset = datasets.CIFAR10(
        root=DATASET_ROOT,
        train=False,
        download=True,
        transform=transform,
    )

    calibration_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
    )

    activation_ranges: dict[str, dict[str, float]] = defaultdict(
        lambda: {"minimum": float("inf"), "maximum": float("-inf")}
    )

    hooks = []

    def register_activation_hook(
        module: nn.Module,
        layer_name: str,
    ) -> None:
        def hook(current_module, inputs, output) -> None:
            current_minimum = float(output.min().item())
            current_maximum = float(output.max().item())

            activation_ranges[layer_name]["minimum"] = min(
                activation_ranges[layer_name]["minimum"],
                current_minimum,
            )
            activation_ranges[layer_name]["maximum"] = max(
                activation_ranges[layer_name]["maximum"],
                current_maximum,
            )

        hooks.append(module.register_forward_hook(hook))

    
    monitored_layers = {
        "conv1": model.features[1],   
        "pool1": model.features[2],
        "conv2": model.features[4],   
        "pool2": model.features[5],
        "conv3": model.features[7],   
        "conv4": model.features[9],   
        "conv5": model.features[11],  
        "pool5": model.features[12],
        "fc6": model.classifier[1],   
        "fc7": model.classifier[3],  
        "fc8": model.classifier[4],   
    }

    for name, layer in monitored_layers.items():
        register_activation_hook(layer, name)

    model.eval()

    with torch.no_grad():
        for batch_index, (images, _) in enumerate(calibration_loader):
            images = images.to(device)
            model(images)

            if batch_index + 1 >= CALIBRATION_BATCHES:
                break

    for hook in hooks:
        hook.remove()

    activation_scales: dict[str, float] = {"input": INPUT_SCALE}

    print()
    print("========================================")
    print("Calibration: activation ranges and scales")
    print("========================================")

    for name, values in activation_ranges.items():
        scale = calculate_symmetric_scale(
            values["minimum"],
            values["maximum"],
        )
        activation_scales[name] = scale

        print(
            f"{name:8s} "
            f"min={values['minimum']:10.6f} "
            f"max={values['maximum']:10.6f} "
            f"scale={scale:.10f}"
        )

    weight_scales: dict[str, float] = {}

    layer_name_by_param_index = {
        "features.0.weight": "conv1",
        "features.3.weight": "conv2",
        "features.6.weight": "conv3",
        "features.8.weight": "conv4",
        "features.10.weight": "conv5",
        "classifier.0.weight": "fc6",
        "classifier.2.weight": "fc7",
        "classifier.4.weight": "fc8",
    }

    print()
    print("========================================")
    print("Calibration: weight ranges and scales")
    print("========================================")

    for param_name, parameter in model.named_parameters():
        if param_name not in layer_name_by_param_index:
            continue

        layer_name = layer_name_by_param_index[param_name]

        minimum = float(parameter.min().item())
        maximum = float(parameter.max().item())
        scale = calculate_symmetric_scale(minimum, maximum)

        weight_scales[layer_name] = scale

        print(
            f"{layer_name:5s} "
            f"min={minimum:10.6f} "
            f"max={maximum:10.6f} "
            f"scale={scale:.10f}"
        )

    return activation_scales, weight_scales


# ============================================================
# QUANTIZATION MODEL
# ============================================================

def quantize_int8(values: torch.Tensor, scale: float) -> np.ndarray:
    quantized = torch.round(values / scale)
    quantized = torch.clamp(quantized, -128, 127)

    return quantized.to(torch.int8).cpu().numpy()


def quantize_bias_int32(
    bias: torch.Tensor,
    input_scale: float,
    weight_scale: float,
) -> np.ndarray:
    bias_scale = input_scale * weight_scale

    quantized = torch.round(bias / bias_scale)

    int32_minimum = -(2**31)
    int32_maximum = (2**31) - 1

    quantized = torch.clamp(quantized, int32_minimum, int32_maximum)

    return quantized.to(torch.int32).cpu().numpy()


def add_quantized_layer(
    exported_data: dict[str, np.ndarray],
    layer_name: str,
    layer: nn.Module,
    input_scale: float,
    output_scale: float,
    weight_scales: dict[str, float],
) -> None:
    weight_scale = weight_scales[layer_name]

    exported_data[f"{layer_name}_weight"] = quantize_int8(
        layer.weight.detach(),
        weight_scale,
    )

    exported_data[f"{layer_name}_bias"] = quantize_bias_int32(
        layer.bias.detach(),
        input_scale,
        weight_scale,
    )

    multiplier = input_scale * weight_scale / output_scale

    exported_data[f"{layer_name}_multiplier"] = np.asarray(
        multiplier,
        dtype=np.float64,
    )

    print(
        f"{layer_name:5s} | "
        f"input scale={input_scale:.10f} | "
        f"weight scale={weight_scale:.10f} | "
        f"output scale={output_scale:.10f} | "
        f"multiplier={multiplier:.10f}"
    )


def main() -> None:
    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    checkpoint = torch.load(
        MODEL_PATH,
        map_location=device,
        weights_only=False,
    )

    model = CifarAlexNet(number_of_classes=10).to(device)
    state_dict = checkpoint.get("model_state_dict", checkpoint)
    model.load_state_dict(state_dict)
    model.eval()

    activation_scales, weight_scales = run_calibration(model, device)

    print()
    print("========================================")
    print("Quantizing layers")
    print("========================================")

    exported_data: dict[str, np.ndarray] = {}

    add_quantized_layer(
        exported_data, "conv1", model.features[0],
        activation_scales["input"], activation_scales["conv1"],
        weight_scales,
    )
    add_quantized_layer(
        exported_data, "conv2", model.features[3],
        activation_scales["pool1"], activation_scales["conv2"],
        weight_scales,
    )
    add_quantized_layer(
        exported_data, "conv3", model.features[6],
        activation_scales["pool2"], activation_scales["conv3"],
        weight_scales,
    )
    add_quantized_layer(
        exported_data, "conv4", model.features[8],
        activation_scales["conv3"], activation_scales["conv4"],
        weight_scales,
    )
    add_quantized_layer(
        exported_data, "conv5", model.features[10],
        activation_scales["conv4"], activation_scales["conv5"],
        weight_scales,
    )
    add_quantized_layer(
        exported_data, "fc6", model.classifier[0],
        activation_scales["pool5"], activation_scales["fc6"],
        weight_scales,
    )
    add_quantized_layer(
        exported_data, "fc7", model.classifier[2],
        activation_scales["fc6"], activation_scales["fc7"],
        weight_scales,
    )
    add_quantized_layer(
        exported_data, "fc8", model.classifier[4],
        activation_scales["fc7"], activation_scales["fc8"],
        weight_scales,
    )

    np.savez(OUTPUT_MODEL_PATH, **exported_data)

    configuration = {
        "input_scale": INPUT_SCALE,
        "activation_scales": activation_scales,
        "weight_scales": weight_scales,
        "layer_order": [
            "conv1", "pool1", "conv2", "pool2", "conv3",
            "conv4", "conv5", "pool5", "fc6", "fc7", "fc8",
        ],
        "input_shape": [3, 32, 32],
        "class_names": CLASS_NAMES,
    }

    with OUTPUT_CONFIG_PATH.open("w", encoding="utf-8") as file:
        json.dump(configuration, file, indent=2)

    print()
    print("========================================")
    print("Quantization export complete")
    print(f"Model:  {OUTPUT_MODEL_PATH}")
    print(f"Config: {OUTPUT_CONFIG_PATH}")
    print("========================================")


if __name__ == "__main__":
    main()